<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session06_embeddings_and_vector_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۶: امبدینگ و جستجوی معنایی

### اهداف این جلسه

- فهم امبدینگ جمله: تبدیل متن به بردار به‌طوری که متن‌های هم‌معنا نزدیک هم باشند
- محاسبه شباهت کسینوسی و مقایسه جستجوی معنایی با جستجوی کلیدواژه‌ای
- ساخت یک موتور جستجوی معنایی فارسی روی اسناد نوین‌کالا
- کار با پایگاه داده برداری: FAISS برای سرعت و Chroma برای ذخیره‌سازی و فیلتر متادیتا
- شناخت خطاهای رایج در استفاده از مدل‌های امبدینگ

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. امبدینگ چیست؟

در جلسه ۱۱ دوره یادگیری عمیق، **امبدینگ کلمه** را دیدیم: هر کلمه یک بردار است و کلمات هم‌معنا بردارهای نزدیک دارند. **امبدینگ جمله (Sentence Embedding)** همین ایده را به یک جمله یا پاراگراف کامل گسترش می‌دهد: کل متن به یک بردار با طول ثابت، مثلاً ۷۶۸ عدد، تبدیل می‌شود.

مدل‌های امبدینگ معمولاً یک **Encoder** از خانواده BERT هستند. خروجی توکن‌ها میانگین‌گیری (Mean Pooling) می‌شود و مدل با **یادگیری متضاد (Contrastive Learning)** روی میلیون‌ها جفت متن هم‌معنا آموزش می‌بیند: بردار پرسش باید به بردار پاسخ درستش نزدیک و از بقیه دور باشد.

این بردارها پایه **جستجوی معنایی (Semantic Search)** و در نتیجه RAG هستند. در RAG برای هر پرسش، مرتبط‌ترین بخش‌های اسناد را با همین بردارها پیدا می‌کنیم.

این نوت‌بوک به مدل زبانی مولد نیاز ندارد. فقط مدل امبدینگ را دانلود می‌کنیم.

</div>

In [ ]:
!pip -q install sentence-transformers faiss-cpu chromadb

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-base")
print("dimension:", embedder.get_sentence_embedding_dimension(),
      "  max tokens:", embedder.max_seq_length)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل `multilingual-e5-base` روی بیش از ۹۰ زبان، از جمله فارسی، آموزش دیده است. این مدل یک قرارداد مهم دارد: پرسش‌ها باید با پیشوند `query: ` و متن اسناد با پیشوند `passage: ` به مدل داده شوند. فراموش کردن پیشوندها کیفیت جستجو را محسوس پایین می‌آورد.

</div>

In [ ]:
def embed_queries(texts):
    return embedder.encode([f"query: {t}" for t in texts], normalize_embeddings=True)


def embed_passages(texts):
    return embedder.encode([f"passage: {t}" for t in texts], normalize_embeddings=True,
                           batch_size=32, show_progress_bar=False)


vector = embed_queries(["هزینه ارسال چقدر است؟"])[0]
print(vector.shape, vector[:6].round(3), "  norm =", np.linalg.norm(vector).round(3))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. شباهت کسینوسی

نزدیکی دو بردار را معمولاً با **شباهت کسینوسی (Cosine Similarity)** می‌سنجیم:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}$$

مقدار ۱ یعنی هم‌جهت و مقدار نزدیک صفر یعنی بی‌ربط. چون بردارها را **نرمال** کرده‌ایم (`normalize_embeddings=True`)، طول همه آن‌ها یک است و شباهت کسینوسی همان **ضرب داخلی** است. این نکته محاسبه را بسیار سریع می‌کند.

</div>

In [ ]:
import matplotlib.pyplot as plt

sentences = [
    "پولم را کی پس می‌دهید؟",
    "بازپرداخت وجه کالای مرجوعی چند روز طول می‌کشد؟",
    "هزینه ارسال سفارش چقدر است؟",
    "برای ارسال بسته چقدر باید پرداخت کنم؟",
    "هوای تهران امروز آلوده است.",
]
E = embed_queries(sentences)
similarity = E @ E.T

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(similarity, cmap="Blues", vmin=0.6, vmax=1)
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_xticks(range(len(sentences)), [f"S{i + 1}" for i in range(len(sentences))])
ax.set_yticks(range(len(sentences)), [f"S{i + 1}" for i in range(len(sentences))])
plt.show()
for i, s in enumerate(sentences, 1):
    print(f"S{i}: {s}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

جمله‌های ۱ و ۲ تقریباً هیچ کلمه مشترکی ندارند، ولی شباهتشان بالاست، چون معنایشان نزدیک است. جمله ۵ با همه کم‌شباهت است.

**نکته:** مقدار مطلق شباهت در مدل‌های مختلف فرق دارد. در E5 حتی متن‌های بی‌ربط هم شباهت حدود ۰٫۷ دارند. چیزی که اهمیت دارد **ترتیب** شباهت‌هاست، نه عدد مطلق. پس آستانه ثابتی مثل «بالای ۰٫۸ یعنی مرتبط» را بدون آزمایش روی داده خودتان به کار نبرید.

## ۳. ساخت یک موتور جستجو روی اسناد نوین‌کالا

اسناد سیاست‌های فروشگاه را دانلود می‌کنیم و هر سند را به پاراگراف‌هایش می‌شکنیم. تقسیم دقیق‌تر متن موضوع جلسه بعد است.

</div>

In [ ]:
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
from llm_utils import load_store_docs

docs = load_store_docs()
passages = []
for source, text in docs.items():
    title = text.split("\n", 1)[0].lstrip("# ").strip()
    for paragraph in text.split("\n\n"):
        paragraph = paragraph.strip()
        if len(paragraph) > 40 and not paragraph.startswith("#"):
            passages.append({"source": source, "title": title, "text": paragraph})

passages = pd.DataFrame(passages)
print(len(passages), "passages from", passages.source.nunique(), "documents")
passages.head(3)

In [ ]:
passage_vectors = embed_passages(passages.text.tolist())
print(passage_vectors.shape)


def semantic_search(query, k=3):
    scores = passage_vectors @ embed_queries([query])[0]
    top = np.argsort(-scores)[:k]
    return passages.iloc[top].assign(score=scores[top])[["score", "title", "text"]]


semantic_search("پولم کی برمی‌گرده؟")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. جستجوی معنایی در برابر جستجوی کلیدواژه‌ای

برای مقایسه، یک جستجوی کلیدواژه‌ای ساده می‌سازیم که فقط تعداد کلمات مشترک پرسش و متن را می‌شمارد.

</div>

In [ ]:
def keyword_search(query, k=3):
    query_words = set(query.split())
    scores = np.array([len(query_words & set(t.split())) for t in passages.text])
    top = np.argsort(-scores)[:k]
    return passages.iloc[top].assign(score=scores[top])[["score", "title", "text"]]


for q in ["پولم کی برمی‌گرده؟", "اگه خونه نباشم چی میشه؟", "کد تخفیف"]:
    print(f"\n=== {q}")
    print("keyword :", keyword_search(q, 1).title.iloc[0])
    print("semantic:", semantic_search(q, 1).title.iloc[0])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

| | جستجوی کلیدواژه‌ای | جستجوی معنایی |
|---|---|---|
| مبنا | کلمات مشترک | نزدیکی معنا |
| مترادف و زبان محاوره | شکست می‌خورد | خوب کار می‌کند |
| کد، شماره و نام خاص (مثلاً NK482913) | دقیق است | ممکن است اشتباه کند |
| نیاز به مدل | ندارد | دارد |

هیچ‌کدام همیشه بهتر نیست. در جلسه ۹ این دو را با **جستجوی ترکیبی (Hybrid Search)** ترکیب می‌کنیم.

## ۵. دیدن فضای امبدینگ

بردارهای ۷۶۸بعدی را با **PCA** به دو بعد می‌بریم تا ساختار آن‌ها را ببینیم. پاراگراف‌های هر سند با یک رنگ نمایش داده می‌شوند.

</div>

In [ ]:
from sklearn.decomposition import PCA

xy = PCA(n_components=2, random_state=0).fit_transform(passage_vectors)
fig, ax = plt.subplots(figsize=(9, 6))
for source, group in passages.groupby("source"):
    ax.scatter(xy[group.index, 0], xy[group.index, 1], label=source.split("_", 1)[1][:-3], s=40)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_title("Passages of the Novin Kala documents (PCA of E5 embeddings)")
plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پاراگراف‌های یک سند معمولاً کنار هم قرار می‌گیرند، و سندهای هم‌موضوع، مثلاً بازگشت کالا و گارانتی، به هم نزدیک‌اند. PCA فقط بخش کوچکی از اطلاعات را در دو بعد نشان می‌دهد، پس از این نمودار برای شهود استفاده کنید، نه نتیجه‌گیری دقیق.

## ۶. FAISS: جستجوی سریع در میلیون‌ها بردار

ضرب داخلی با همه بردارها برای چند هزار متن کافی است، ولی برای میلیون‌ها متن کند می‌شود. کتابخانه **FAISS** از Meta برای جستجوی سریع بردار ساخته شده است. ساده‌ترین شاخص آن، `IndexFlatIP`، همان جستجوی کامل با ضرب داخلی است:

</div>

In [ ]:
import faiss

index = faiss.IndexFlatIP(passage_vectors.shape[1])
index.add(passage_vectors.astype("float32"))

scores, ids = index.search(embed_queries(["نصب کولر گازی"]).astype("float32"), 3)
for score, i in zip(scores[0], ids[0]):
    print(f"{score:.3f}  {passages.title[i]}: {passages.text[i][:70]}...")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برای مقیاس‌های بزرگ، FAISS شاخص‌های **تقریبی (Approximate Nearest Neighbor)** دارد که کمی از دقت را با سرعت بسیار بیشتر عوض می‌کنند. یکی از رایج‌ترین آن‌ها **HNSW** است: گرافی چندلایه از همسایه‌ها که جستجو در آن مثل پریدن از شهری به شهر دیگر روی نقشه است.

آزمایش زیر با ۱۰۰ هزار بردار مصنوعی تفاوت را نشان می‌دهد. بردارها را طوری می‌سازیم که مثل امبدینگ‌های واقعی ساختار داشته باشند، یعنی روی یک زیرفضای کم‌بعد قرار بگیرند. بردارهای کاملاً تصادفی بدترین حالت برای جستجوی تقریبی‌اند و در عمل پیش نمی‌آیند. ساخت شاخص HNSW حدود یک دقیقه طول می‌کشد.

</div>

In [ ]:
import time

rng = np.random.default_rng(0)
big = (rng.normal(size=(100_000, 48)) @ rng.normal(size=(48, 768))).astype("float32")
faiss.normalize_L2(big)
queries = big[:100] + 0.02 * rng.normal(size=(100, 768)).astype("float32")
faiss.normalize_L2(queries)

flat = faiss.IndexFlatIP(768)
flat.add(big)
hnsw = faiss.IndexHNSWFlat(768, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.add(big)

start = time.time()
_, exact = flat.search(queries, 10)
print(f"Flat (exact)        {(time.time() - start) * 10:6.2f} ms/query")

for ef in [16, 64]:
    hnsw.hnsw.efSearch = ef
    start = time.time()
    _, found = hnsw.search(queries, 10)
    ms = (time.time() - start) * 10
    recall = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(found, exact)])
    print(f"HNSW efSearch={ef:<4} {ms:6.2f} ms/query   recall@10 vs exact: {recall:.0%}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پارامتر `efSearch` تعداد گره‌هایی است که در هر جستجو بررسی می‌شوند. مقدار بیشتر یعنی دقت بیشتر و سرعت کمتر. پایگاه‌های داده برداری همین موازنه را با پارامترهای مشابه در اختیار شما می‌گذارند.

## ۷. Chroma: پایگاه داده برداری

FAISS فقط بردار نگه می‌دارد. در برنامه واقعی به متن، **متادیتا**، به‌روزرسانی و ذخیره روی دیسک هم نیاز داریم. **پایگاه داده برداری (Vector Database)** این امکانات را می‌دهد. Chroma یک پایگاه داده سبک و متن‌باز است که بدون سرور جداگانه کار می‌کند. نمونه‌های دیگر: Qdrant، Milvus، Weaviate و افزونه pgvector برای PostgreSQL.

بردارها را خودمان می‌سازیم و به Chroma می‌دهیم تا کنترل کامل روی مدل و پیشوندها داشته باشیم.

</div>

In [ ]:
import chromadb

chroma = chromadb.PersistentClient(path="chroma_db")
collection = chroma.get_or_create_collection("novinkala", metadata={"hnsw:space": "cosine"})
collection.upsert(
    ids=[f"p{i}" for i in range(len(passages))],
    documents=passages.text.tolist(),
    embeddings=passage_vectors.tolist(),
    metadatas=passages[["source", "title"]].to_dict("records"),
)
print("stored:", collection.count())

result = collection.query(query_embeddings=embed_queries(["قسطی می‌شه خرید کرد؟"]).tolist(), n_results=2)
for doc, meta, dist in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
    print(f"{1 - dist:.3f}  [{meta['title']}] {doc[:80]}...")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**فیلتر متادیتا** جستجو را به بخشی از اسناد محدود می‌کند. مثلاً فقط در اسناد ارسال و بازگشت کالا:

</div>

In [ ]:
result = collection.query(
    query_embeddings=embed_queries(["چند روز طول می‌کشد؟"]).tolist(), n_results=3,
    where={"source": {"$in": ["01_shipping.md", "02_returns.md"]}},
)
for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print(f"[{meta['source']}] {doc[:90]}...")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۸. خطاهای رایج

| خطا | پیامد | راه‌حل |
|---|---|---|
| فراموش کردن پیشوند `query:` و `passage:` در E5 | افت کیفیت جستجو | کپسوله کردن در تابع |
| مدل متفاوت برای ساخت شاخص و پرسش | نتایج بی‌معنی | ذخیره نام مدل کنار شاخص |
| متن بلندتر از ۵۱۲ توکن | انتهای متن بی‌صدا بریده می‌شود | تقسیم متن (جلسه ۷) |
| آستانه ثابت شباهت | حذف نتایج درست یا ورود نتایج بی‌ربط | تنظیم آستانه با داده ارزیابی |
| فرض «معنایی همیشه بهتر است» | شکست در کدها و نام‌های خاص | جستجوی ترکیبی (جلسه ۹) |

### مدل‌های دیگر

| مدل | ویژگی |
|---|---|
| `intfloat/multilingual-e5-base` و `-large` | چندزبانه، رایگان، کیفیت خوب برای فارسی |
| `BAAI/bge-m3` | چندزبانه، ورودی تا ۸۱۹۲ توکن، کیفیت بالاتر و کندتر |
| `nomic-embed-text` در Ollama | اجرا با Ollama و API سازگار با OpenAI |
| `text-embedding-3-small` از OpenAI | سرویس پولی، بدون نیاز به GPU |

با API سازگار با OpenAI، امبدینگ گرفتن یک خط کد است. برای Ollama ابتدا `ollama pull nomic-embed-text` را اجرا کنید:

```python
vectors = client.embeddings.create(model="nomic-embed-text", input=["متن اول", "متن دوم"])
```

## ۹. جمع‌بندی

- امبدینگ جمله، متن را به برداری تبدیل می‌کند که متن‌های هم‌معنا را نزدیک هم قرار می‌دهد.
- با بردارهای نرمال، شباهت کسینوسی همان ضرب داخلی است. ترتیب شباهت‌ها مهم است، نه عدد مطلق.
- جستجوی معنایی مترادف و زبان محاوره را می‌فهمد، ولی در کدها و نام‌های خاص ضعیف است.
- FAISS برای سرعت در مقیاس بزرگ است و پایگاه داده برداری مثل Chroma متن، متادیتا و ذخیره‌سازی را اضافه می‌کند.
- قرارداد مدل، مثل پیشوندهای E5 و حداکثر طول ورودی، را رعایت کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. پیشوندهای `query:` و `passage:` را حذف کنید و نتیجه پنج پرسش را با حالت درست مقایسه کنید. شباهت‌ها و ترتیب نتایج چه تغییری کرد؟
۲. ده پرسش محاوره‌ای درباره سیاست‌های نوین‌کالا بنویسید و برای هر کدام بررسی کنید سند درست در سه نتیجه اول جستجوی معنایی و کلیدواژه‌ای هست یا نه. کدام بهتر بود؟
۳. مدل `BAAI/bge-m3` را جایگزین کنید (این مدل پیشوند لازم ندارد) و تمرین ۲ را تکرار کنید. کیفیت و سرعت را مقایسه کنید.
۴. در Chroma یک فیلد متادیتای جدید به نام `topic` با مقدارهای «مالی» و «لجستیک» بسازید و جستجویی بنویسید که فقط در اسناد مالی جستجو کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** پیشوندهای `query:` و `passage:` را حذف کنید و نتیجه پنج پرسش را با حالت درست مقایسه کنید. شباهت‌ها و ترتیب نتایج چه تغییری کرد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** ده پرسش محاوره‌ای درباره سیاست‌های نوین‌کالا بنویسید و برای هر کدام بررسی کنید سند درست در سه نتیجه اول جستجوی معنایی و کلیدواژه‌ای هست یا نه. کدام بهتر بود؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** مدل `BAAI/bge-m3` را جایگزین کنید (این مدل پیشوند لازم ندارد) و تمرین ۲ را تکرار کنید. کیفیت و سرعت را مقایسه کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** در Chroma یک فیلد متادیتای جدید به نام `topic` با مقدارهای «مالی» و «لجستیک» بسازید و جستجویی بنویسید که فقط در اسناد مالی جستجو کند.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید